# Stardox Email Scraper

Takes a list of GitHub usernames and scrapes their public email addresses using a headless browser.

**How it works:**
1. Spins up headless Chrome
2. Visits each user's GitHub profile
3. Looks for email in the profile sidebar (JS-rendered)
4. If no email on profile, checks their commit history (.patch files)
5. Outputs username:email pairs as a downloadable CSV

In [ ]:
# Install dependencies
!pip install -q selenium webdriver-manager beautifulsoup4 lxml pandas tqdm

In [ ]:
import re
import time
import pandas as pd
from tqdm.notebook import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

EMAIL_RE = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
NOREPLY_PATTERNS = ['noreply', 'users.noreply.github.com']


def start_browser(github_cookie=None):
    """Launch headless Chrome, optionally with GitHub session cookie."""
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(20)

    if github_cookie:
        # Visit github.com first to set the domain, then inject cookie
        driver.get('https://github.com')
        time.sleep(1)
        driver.add_cookie({
            'name': 'user_session',
            'value': github_cookie,
            'domain': '.github.com',
            'path': '/',
            'secure': True,
        })
        driver.refresh()
        time.sleep(2)
        print('Browser started with GitHub session!')
    else:
        print('Browser started (anonymous — profile emails may be hidden)')

    return driver


def is_valid_email(email):
    if not email:
        return False
    email_lower = email.lower()
    for pattern in NOREPLY_PATTERNS:
        if pattern in email_lower:
            return False
    return True


def scrape_email_from_profile(driver, username):
    """Visit GitHub profile and look for email in the sidebar."""
    try:
        driver.get(f'https://github.com/{username}')
        time.sleep(3)

        # Check vcard-details for email
        try:
            vcard = driver.find_element(By.CSS_SELECTOR, 'ul.vcard-details')
            items = vcard.find_elements(By.CSS_SELECTOR, 'li.vcard-detail')
            for item in items:
                # Check text content
                text = item.text.strip()
                match = EMAIL_RE.search(text)
                if match and is_valid_email(match.group()):
                    return match.group()

                # Check mailto links
                links = item.find_elements(By.TAG_NAME, 'a')
                for link in links:
                    href = link.get_attribute('href') or ''
                    if href.startswith('mailto:'):
                        email = href.replace('mailto:', '').strip()
                        if is_valid_email(email):
                            return email

                # Check itemprop="email"
                email_els = item.find_elements(By.CSS_SELECTOR, '[itemprop="email"]')
                for el in email_els:
                    text = el.text.strip()
                    if is_valid_email(text):
                        return text
                    link = el.find_elements(By.TAG_NAME, 'a')
                    if link:
                        t = link[0].text.strip()
                        if is_valid_email(t):
                            return t
        except Exception:
            pass

        # Broader search in page source
        source = driver.page_source
        emails = EMAIL_RE.findall(source)
        for email in emails:
            if is_valid_email(email):
                return email

    except Exception:
        pass

    return None


def scrape_email_from_commits(driver, username):
    """Get email from user's commit .patch files."""
    try:
        driver.get(f'https://github.com/{username}?tab=repositories&type=source')
        time.sleep(2)

        repo_links = driver.find_elements(By.CSS_SELECTOR, 'a[itemprop="name codeRepository"]')
        repo_names = [link.text.strip() for link in repo_links[:3]]

        if not repo_names:
            return None

        for repo_name in repo_names:
            try:
                driver.get(f'https://github.com/{username}/{repo_name}/commits?author={username}')
                time.sleep(2)

                commit_links = driver.find_elements(By.CSS_SELECTOR, f'a[href*="/{username}/{repo_name}/commit/"]')

                for commit_link in commit_links[:5]:
                    href = commit_link.get_attribute('href')
                    if not href or '/commit/' not in href:
                        continue

                    label = commit_link.get_attribute('aria-label') or ''
                    text = commit_link.text or ''
                    if 'merge' in label.lower() or 'merge' in text.lower():
                        continue

                    driver.get(href + '.patch')
                    time.sleep(1)

                    page_text = driver.page_source

                    from_match = re.search(r'From:.*?<([^>]+@[^>]+)>', page_text)
                    if from_match:
                        email = from_match.group(1)
                        if is_valid_email(email):
                            return email

                    emails = EMAIL_RE.findall(page_text[:3000])
                    for email in emails:
                        if is_valid_email(email):
                            return email

            except Exception:
                continue

    except Exception:
        pass

    return None


def scrape_email(driver, username):
    """Try profile first, then commits."""
    email = scrape_email_from_profile(driver, username)
    if email:
        return email
    return scrape_email_from_commits(driver, username)


print('Functions loaded. Ready to scrape.')

In [ ]:
# ===========================================
# GITHUB SESSION COOKIE (required to see profile emails)
#
# How to get it:
# 1. Log into github.com in your browser
# 2. Open DevTools (F12) -> Application -> Cookies -> github.com
# 3. Find the "user_session" cookie and copy its value
# ===========================================

GITHUB_COOKIE = ""  # paste your user_session cookie value here

# ===========================================
# PASTE YOUR USERNAMES BELOW (one per line)
# ===========================================

usernames_input = """
bnbarak
torvalds
sindresorhus
"""

# ===========================================

usernames = [u.strip() for u in usernames_input.strip().split('\n') if u.strip()]
print(f'Loaded {len(usernames)} usernames')
if GITHUB_COOKIE:
    print('GitHub cookie provided — will see profile emails')
else:
    print('No GitHub cookie — will only get emails from commit history')

In [ ]:
# Run the scraper
driver = start_browser(github_cookie=GITHUB_COOKIE if GITHUB_COOKIE else None)
results = []
found_count = 0

try:
    for username in tqdm(usernames, desc='Scraping emails'):
        email = scrape_email(driver, username)
        results.append({'username': username, 'email': email})

        if email:
            found_count += 1
            print(f'  ✓ {username} -> {email}')
        else:
            print(f'  ✗ {username} -> not found')

        time.sleep(1)  # be polite
finally:
    driver.quit()

print(f'\nDone! Found {found_count}/{len(usernames)} emails')

In [ ]:
# Results
df = pd.DataFrame(results)
print(f'Total: {len(df)}')
print(f'With email: {df["email"].notna().sum()}')
print(f'Without email: {df["email"].isna().sum()}')
print()

# Show all results
display(df)

# Save and download CSV
csv_filename = 'stargazer_emails.csv'
df.to_csv(csv_filename, index=False)

from google.colab import files
files.download(csv_filename)
print(f'\nDownloading {csv_filename}...')